# Inicio

## Requirements

In [ ]:
# ============================================================
# Instalación de librerías necesarias desde el notebook
# ============================================================
# import importlib.util
# import subprocess
# import sys

# REQUIRED_PACKAGES = {
#     "numpy": "numpy",
#     "ydata-profiling": "ydata-profiling",
#     "scikit-surprise": "scikit-surprise",
#     # "rarfile": "rarfile",
# }

# missing = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items()
#            if importlib.util.find_spec(import_name) is None]

# if missing:
#     print("Instalando librerías faltantes:", missing)
#     subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
# else:
#     print("Todas las librerías necesarias ya están instaladas.")

Instalando librerías faltantes: ['ydata-profiling', 'scikit-surprise']


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import gdown
file_id='10onBXp0Krg7RODnT2q2JYSBV_O9HI2ZS'
file_url = f'https://drive.google.com/uc?id={file_id}'
output_path = 'requirements.txt'
gdown.download(file_url, output_path, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=10onBXp0Krg7RODnT2q2JYSBV_O9HI2ZS
To: /content/requirements.txt
100%|██████████| 52.0/52.0 [00:00<00:00, 79.4kB/s]


'requirements.txt'

In [2]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 14.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 125.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.6 MB/s eta 0:00:00
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554968 sha256=026dd92668c1b13a129c9770d17b

## Imports

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from surprise import SVD, SVDpp
from surprise import Reader
from surprise import Dataset
from surprise import accuracy
from surprise.model_selection import GridSearchCV, train_test_split

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics.pairwise import cosine_similarity

from concurrent.futures import ThreadPoolExecutor

import json
import time
import os
import tarfile
import random
import warnings
import duckdb
import multiprocessing
import joblib
from datetime import datetime

from ydata_profiling import ProfileReport # Perfilamiento de datos

%matplotlib inline

/tmp/ipykernel_5029/2243662064.py:31: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport # Perfilamiento de datos


In [2]:
# Para garantizar reproducibilidad en resultados, se define la semilla global
seed = 10
random.seed(seed)
np.random.seed(seed)

# Configuración global
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option('display.float_format', '{:.3f}'.format) # Configuración global de presentación de .3 decimales en resul

def timer(start_time=None):
  if not start_time:
    start_time = datetime.now()
    return start_time
  elif start_time:
    thour, temp_sec = divmod((datetime.now() - start_time).total_seconds(), 3600)
    tmin, tsec = divmod(temp_sec, 60)
    print('\n Time taken: %i hour(s) %i minute(s) and %s second(s).' % (thour, tmin, round(tsec, 2)))

# Carga de datos

In [3]:
tar_path = "/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Sistemas_recomendacion/Taller 2/yelp_dataset.tar"

# Folder where files will be extracted
extract_path = "/content/yelp_dataset"
os.makedirs(extract_path, exist_ok=True)

with tarfile.open(tar_path, "r") as tar:
    tar.extractall(path=extract_path)
print(f"Files extracted to: {extract_path}")

Files extracted to: /content/yelp_dataset


## Business

In [ ]:
# business_path='/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/yelp_academic_dataset_business.json'
df_business = pd.read_json('/content/yelp_dataset/yelp_academic_dataset_business.json', lines=True)
print(df_business.shape)
df_business.head()

(150346, 14)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.427,-119.711,5.000,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551,-90.336,3.000,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223,-110.880,3.500,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.956,-75.156,4.000,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338,-75.472,4.500,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


## Users

In [5]:
df_user = pd.read_json('/content/yelp_dataset/yelp_academic_dataset_user.json', lines=True)
print(df_user.shape)
df_user.head()

(1987897, 22)


,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,average_stars,compliment_hot,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,3.910,250,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,3.740,1145,264,184,157,251,1847,7054,3131,3131,1521,1946
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013","LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA...",52,3.320,89,13,10,17,3,66,96,119,119,35,18
3,SZDeASXq7o05mMNLshsdIA,Gwen,224,2005-11-29 04:38:33,512,330,299,"2009,2010,2011","enx1vVPnfdNUdPho6PH_wg, 4wOcvMLtU6a9Lslggq74Vg...",28,4.270,24,4,1,6,2,12,16,26,26,10,9
4,hA5lMy-EnncsH4JoR-hFGQ,Karen,79,2007-01-05 19:40:59,29,15,7,,"PBK4q9KEEBHhFvSXCUirIw, 3FWPpM7KU1gXeOM_ZbYMbA...",1,3.540,1,1,0,0,0,1,1,0,0,0,0


In [20]:
df_user_sampled=df_user.sample(frac=0.8, random_state=seed)
df_user_sampled.shape

(1590318, 22)

## Reviews

In [ ]:
df_review = pd.read_json('/content/yelp_dataset/yelp_academic_dataset_review.json', lines=True)
print(df_review.shape)
# df_review.head()

(6990280, 9)


In [21]:
# Only load target users
target_users=df_user_sampled['user_id'].drop_duplicates().tolist()
pd.DataFrame({"user_id": target_users}).to_csv("target_users.csv", index=False)

query = f"""
COPY (
    SELECT r.*
    FROM read_json_auto('/content/yelp_dataset/yelp_academic_dataset_review.json') r
    INNER JOIN read_csv_auto('target_users.csv') u
    ON r.user_id = u.user_id
)
TO 'filtered_reviews.jsonl'
(FORMAT JSON);
"""
duckdb.sql(query)

df_review = pd.read_json("filtered_reviews.jsonl", lines=True)
print(df_review.shape)
df_review.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(5595630, 9)


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,Nk2LwiH_j7G6DZqnDppGZA,-R6dgi_HAYvj9e4FH7NTlQ,eaDZlSuVS0EY67Ke6pRP6Q,4,1,0,1,I love this Chinatown restaurant. The decor is...,2009-11-04 21:44:52
1,M2Aok4csmsQEC5cDNT6xBQ,S4rg_rw5DtDH2cTQJsgeOg,lwdkX7KcibM4mDqpDfK7JA,5,2,0,0,One of my new favorite restaurants! \n\nThe sa...,2013-08-18 18:46:52
2,g91jIy7BtckLGpdeSFyxCA,EMJV9rib660I4RpMsbzWbg,pym7c6ZFEtmoH16xN2ApBg,4,5,0,4,Crawfish beignet and a Boudreaux pizza... you ...,2014-05-28 17:23:00
3,sDb4s8tYlfsxYbNf_V9BqQ,P5T4eBDKKUiic9ZqRa_PJQ,uO39--k_hrCFgZh-Bl8m8A,4,0,0,0,El Fortschon y Sopes:Chorizo. Delicious. No s...,2013-05-13 18:07:42
4,b6zj8-dqnhEd86ldwyNwow,AN5-bfEnhLc7g5BrjDi4pg,-lhHSR1Aws4CyYC1XOPVTw,2,0,1,0,"Love the food, but.... Perhaps this experience...",2013-03-24 23:43:50


## Profiling

In [ ]:
# Perfilamiento
profiling=0
if profiling:
  sample=.1
  output_path='/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/'

  #Business
  df_sampled=df_business.sample(frac=sample, random_state=seed)
  profile_checkin = ProfileReport(df_sampled)
  profile_checkin.to_file(output_file=output_path+'profile_business.html')
  # #User
  df_sampled=df_user.sample(frac=sample/10, random_state=seed)
  profile_checkin = ProfileReport(df_sampled)
  profile_checkin.to_file(output_file=output_path+'profile_user.html')
  # #Review
  df_sampled=df_review.sample(frac=sample/10, random_state=seed)
  profile_checkin = ProfileReport(df_sampled)
  profile_checkin.to_file(output_file=output_path+'profile_review.html')

# Modelo SVD factorizado

In [22]:
reader = Reader( rating_scale = ( 1, 5 ) )
data = Dataset.load_from_df(df_review[['user_id', 'business_id', 'stars']], reader)
train_set, test_set = train_test_split(data, test_size=0.2, random_state=seed)
print("# user_id:", train_set.n_users,' '*5,"#business_id :", train_set.n_items,' '*5,"# reviews:", train_set.n_ratings,' '*5,"# test reviews:", len(test_set))

# user_id: 1397621       #business_id : 150226       # reviews: 4476504       # test reviews: 1119126


In [9]:
search=True
if search:
  param_grid = {
      'n_factors': [25,50,100,150],
      'n_epochs': [20, 30, 50],
      'lr_all': [0.005],
      'reg_all': [0.05],
      'random_state': [seed],
  }

## SVD

### GS

In [ ]:
if search:
  gs_svd = GridSearchCV(SVD, param_grid, measures=['rmse','mae'], cv=3, n_jobs=-1)

In [ ]:
%%time
if search:
  start_time = timer(None)
  gs_svd.fit(data)
  timer(start_time)

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 6.68 µs


In [ ]:
if search:
  gs_svd_results=pd.DataFrame(gs_svd.cv_results).sort_values(by='rank_test_rmse')
  gs_svd_results.drop(columns='params').head(10)

In [ ]:
if search:
  best_params = gs_svd.best_params['rmse']
  best_score  = gs_svd.best_score['rmse']

  # Ver los mejores parámetros
  print("Mejores parámetros encontrados: ", best_params)
  print("Mejor RMSE: ", best_score)



In [ ]:
%%time
if search:
  svd_best_model = gs_svd.best_estimator['rmse']
  svd_best_model.fit(train_set)

  # Evaluar modelo en test
  print('Evaluación de Test')
  predictions = svd_best_model.test(test_set)
  rmse = accuracy.rmse(predictions)
  mae = accuracy.mae(predictions)

CPU times: user 3 µs, sys: 1e+03 ns, total: 4 µs
Wall time: 28.1 µs


### Final model

In [ ]:
# best params
%%time
final_svd=SVD(
    n_factors = 25,
    n_epochs = 30,
    lr_all = 0.005,
    reg_all = 0.05,
    random_state = seed,
)
final_svd.fit(train_set)

print('Evaluación de Test')
predictions = final_svd.test(test_set)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

Evaluación de Test
RMSE: 1.2730
MAE:  1.0146
CPU times: user 3min 45s, sys: 1.56 ms, total: 3min 45s
Wall time: 3min 46s


### Guardar modelo

In [ ]:
# Guardar modelo
svd_saved_path='/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/models/'
# joblib.dump(final_svd,svd_saved_path+'model_SVD_09.joblib')
joblib.dump(final_svd,'model_SVD_100.joblib')

['model_SVD_100.joblib']

## SVD++

### GS

In [10]:
if search:
  gs_svdpp = GridSearchCV(SVDpp, param_grid, measures=['rmse','mae'], cv=3, n_jobs=-1)

In [11]:
%%time
if search:
  start_time = timer(None)
  gs_svdpp.fit(data)
  timer(start_time)


 Time taken: 1 hour(s) 5 minute(s) and 7.08 second(s).
CPU times: user 5min 20s, sys: 25.2 s, total: 5min 45s
Wall time: 1h 5min 7s


In [12]:
if search:
  gs_svdpp_results=pd.DataFrame(gs_svdpp.cv_results).sort_values(by='rank_test_rmse')
  display(gs_svdpp_results.drop(columns='params').head(10))

In [13]:
if search:
  best_params = gs_svdpp.best_params['rmse']
  best_score  = gs_svdpp.best_score['rmse']

  # Ver los mejores parámetros
  print("Mejores parámetros encontrados: ", best_params)
  print("Mejor RMSE: ", best_score)

Mejores parámetros encontrados:  {'n_factors': 25, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.05, 'random_state': 10}
Mejor RMSE:  1.3232671115750885


In [14]:
%%time
if search:
  svdpp_best_model = gs_svdpp.best_estimator['rmse']
  svdpp_best_model.fit(train_set)

  # Evaluar modelo en test
  print('Evaluación de Test')
  predictions = svdpp_best_model.test(test_set)
  rmse = accuracy.rmse(predictions)
  mae = accuracy.mae(predictions)

Evaluación de Test
RMSE: 1.3132
MAE:  1.0613
CPU times: user 5min 15s, sys: 373 ms, total: 5min 15s
Wall time: 5min 15s


### Final model

In [23]:
# best params
%%time
final_svdpp=SVDpp(
    n_factors = 25,
    n_epochs = 30,
    lr_all = 0.005,
    reg_all = 0.05,
    random_state = seed,
)
final_svdpp.fit(train_set)

print('Evaluación de Test')
predictions = final_svdpp.test(test_set)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

Evaluación de Test
RMSE: 1.2839
MAE:  1.0285
CPU times: user 21min 43s, sys: 2.13 s, total: 21min 45s
Wall time: 21min 45s


### Guardar modelo

In [ ]:
# Guardar modelo
svdpp_saved_path='/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/models/'
# joblib.dump(final_svdpp,svdpp_saved_path+'model_SVDpp_09.joblib')
joblib.dump(final_svdpp,'model_SVDpp_100.joblib')

['model_SVDpp_100.joblib']

# Load of models

In [ ]:
load_model=joblib.load(svdpp_saved_path+'model_SVDpp.joblib')

In [ ]:
print('(n_users, n_factors)',load_model.pu.shape)
print('(n_items, n_factors)',load_model.qi.shape)
print('Parametros:',
  '\n  n_factors:', load_model.n_factors,
  '\n  n_epochs:', load_model.n_epochs,
      )

# End